In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

file_path = "Clean_Dataset.csv"

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "shubhambathwal/flight-price-prediction",
    file_path,
)

df = df.rename(columns={"Unnamed: 0": "ID"})


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="duration",
    y="price",
    hue="class",
    alpha=0.3,
    s=15,
    ax=ax,
)
sns.regplot(
    data=df[df["class"] == "Economy"],
    x="duration",
    y="price",
    scatter=False,
    order=2,
    ci=None,
    color="tab:blue",
    ax=ax,
)
sns.regplot(
    data=df[df["class"] == "Business"],
    x="duration",
    y="price",
    scatter=False,
    order=2,
    ci=None,
    color="tab:orange",
    ax=ax,
)

ax.set_title("Ticket Price vs. Flight Duration by Class")
ax.set_xlabel("Duration (hours)")
ax.set_ylabel("Price")
plt.tight_layout()
plt.show()


In [ ]:
bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
df["duration_bucket"] = pd.cut(df["duration"], bins)

bucket_price = (
    df.groupby(["duration_bucket", "class"], observed=True)["price"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=bucket_price,
    x="duration_bucket",
    y="price",
    hue="class",
    ax=ax,
)

ax.set_title("Average Ticket Price by Duration Bucket and Class")
ax.set_xlabel("Duration Bucket (hours)")
ax.set_ylabel("Average Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
stops_pct = (
    pd.crosstab(df["duration_bucket"], df["stops"], normalize="index")[
        ["zero", "one", "two_or_more"]
    ]
    * 100
)

fig, ax = plt.subplots(figsize=(12, 6))
stops_pct.plot(kind="bar", stacked=True, ax=ax)

ax.set_title("Share of Flights by Number of Stops within Each Duration Bucket")
ax.set_xlabel("Duration Bucket (hours)")
ax.set_ylabel("Percentage of Flights")
ax.legend(title="Stops")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
stops_class_price = (
    df.groupby(["stops", "class"], observed=True)["price"]
    .mean()
    .unstack("class")[["Business", "Economy"]]
    .reindex(["zero", "one", "two_or_more"])
)

fig, ax = plt.subplots(figsize=(10, 6))
stops_class_price.plot(
    kind="bar",
    stacked=True,
    color=["tab:orange", "tab:blue"],
    ax=ax,
)

ax.set_title("Average Ticket Price by Number of Stops and Class")
ax.set_xlabel("Stops")
ax.set_ylabel("Average Price")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
days_left_bins = [0, 3, 7, 15, 30, 50]
df["days_left_bucket"] = pd.cut(df["days_left"], days_left_bins)

days_left_price = (
    df.groupby(["days_left_bucket", "class"], observed=True)["price"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=days_left_price,
    x="days_left_bucket",
    y="price",
    hue="class",
    hue_order=["Economy", "Business"],
    ax=ax,
)

ax.set_title("Average Ticket Price by Days-Left Bucket and Class")
ax.set_xlabel("Days Left Before Departure")
ax.set_ylabel("Average Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
